In [2]:
import pandas as pd

file_path = "/home/mudda/cluster2011/final_sorted.csv"

# Read only first 10 rows safely
df = pd.read_csv(file_path, nrows=1)

print(df)


      timestamp   job_id  task_index  machine_id  cpu_rate  maximum_cpu_rate  \
0  436800000000  3418309           0  4155527081  0.001637           0.01593   

     cpi       mai  canonical_memory_usage  assigned_memory_usage  \
0  2.355  0.006868                 0.06775                0.07568   

   maximum_memory_usage  unmapped_page_cache  total_page_cache  disk_io_time  \
0               0.06775             0.000978          0.001339      0.000003   

   maximum_disk_io_time  local_disk_space_usage  failed  
0              0.000143                0.000188       0  


In [2]:
import tensorflow as tf
print("TF version:", tf.__version__)

from tensorflow import keras
print("Keras version:", keras.__version__)


TF version: 2.20.0
Keras version: 3.13.2


In [2]:
import zipfile
import json
import shutil
import os

model_path = "my_best_model2.keras"
fixed_model_path = "my_best_model2_fixed.keras"

# Extract original .keras file
with zipfile.ZipFile(model_path, 'r') as zip_ref:
    zip_ref.extractall("temp_model")

# Load config
with open("temp_model/config.json", "r") as f:
    config = json.load(f)

# Function to recursively fix axis issue
def fix_axis(obj):
    if isinstance(obj, dict):
        for key, value in obj.items():
            if key == "axis" and isinstance(value, list):
                obj[key] = value[0]
            else:
                fix_axis(value)
    elif isinstance(obj, list):
        for item in obj:
            fix_axis(item)

fix_axis(config)

# Save fixed config
with open("temp_model/config.json", "w") as f:
    json.dump(config, f)

# Repack into new .keras file
with zipfile.ZipFile(fixed_model_path, 'w') as zip_ref:
    for foldername, subfolders, filenames in os.walk("temp_model"):
        for filename in filenames:
            file_path = os.path.join(foldername, filename)
            zip_ref.write(file_path, os.path.relpath(file_path, "temp_model"))

# Clean temp folder
shutil.rmtree("temp_model")

print("✅ Model fixed and saved as:", fixed_model_path)


✅ Model fixed and saved as: my_best_model2_fixed.keras


In [4]:
from tensorflow import keras

model = keras.models.load_model("my_best_model2_fixed.keras", compile=False)
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         2,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,185 (51.50 KB)

 Trainable params: 12,801 (50.00 KB)

 Non-trainable params: 384 (1.50 KB)

In [1]:
import tensorflow as tf

# Path to your saved model
model_path = "/home/mudda/my_best_model2.keras"   # change this path

# Load model
model = tf.keras.models.load_model(model_path)

# Print model summary
model.summary()


2026-02-16 23:23:56.246087: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-16 23:23:56.280472: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-16 23:23:57.056179: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1771264438.058378   47705 gpu_device.cc:2020] Created device /job:localhost/rep

TypeError: <class 'keras.src.models.sequential.Sequential'> could not be deserialized properly. Please ensure that components that are Python object instances (layers, models, etc.) returned by `get_config()` are explicitly deserialized in the model's `from_config()` method.

config={'module': 'keras', 'class_name': 'Sequential', 'config': {'name': 'sequential', 'layers': [{'module': 'keras.layers', 'class_name': 'InputLayer', 'config': {'batch_input_shape': [None, 15], 'dtype': 'float32', 'sparse': False, 'ragged': False, 'name': 'input_1'}, 'registered_name': None}, {'module': 'keras.layers', 'class_name': 'Dense', 'config': {'name': 'dense', 'trainable': True, 'dtype': 'float32', 'units': 128, 'activation': 'relu', 'use_bias': True, 'kernel_initializer': {'module': 'keras.initializers', 'class_name': 'GlorotUniform', 'config': {'seed': None}, 'registered_name': None}, 'bias_initializer': {'module': 'keras.initializers', 'class_name': 'Zeros', 'config': {}, 'registered_name': None}, 